# Create test/train splits

In [ ]:
# Load the modules that we'll be using
import sqlite3 #to connect the database file
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, precision_score, ConfusionMatrixDisplay
from sklearn.preprocessing import OneHotEncoder
from imblearn.over_sampling import SMOTE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

In [ ]:
#Connect the database file
conn = sqlite3.connect('Assignment2025S2.sqlite')

query_vegetation = "SELECT * FROM train;"
query_unseen_dataset = "SELECT * FROM test;"

#Test/train splits
vegetation = pd.read_sql(query_vegetation, conn)
X = vegetation.drop('class', axis=1)
y = vegetation["class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=15)

unseen_dataset = pd.read_sql(query_unseen_dataset, conn)
X_unseen = unseen_dataset.drop('class', axis=1)

conn.close()

In [ ]:
# Double check the shape of each of the above is as expected
X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_unseen.shape

# Data preparation

Before classification, you make sure out data is clean for ML algorithm to classify. Also, In this dataset, you should clean X_train, X_test and X_unseen based on X_train.

## Identify and remove irrelevant attributes

In [ ]:
#Remove irrelevant attribute
X_train.drop("index", axis=1, inplace=True)
X_test.drop("index", axis=1, inplace=True)
X_unseen.drop("index", axis=1, inplace=True)
X_train.head()

## Detect and handle missing entries

### Detect missing entries
1. Check the missing entries
2.   For columns which have a lot of missing data(more
than 50%), remove the columns

In [ ]:
# count the number of null entries in X_train(X_train)
# For columns which have a lot of missing data(>50%), remove the column (Latitude)
# And also apply this to X_test and X_unseen

# compute the fraction of missing values
X_test.isnull().sum()/len(X_train)

In [ ]:
# Delete the column "Latitude" in X_train, X_test and X_unseen

X_train.drop("Latitude", axis=1, inplace=True)
#do the same on the X_test and X_unseen
X_test.drop("Latitude", axis=1, inplace=True)
X_unseen.drop("Latitude", axis=1, inplace=True)

### Select suitable data types for attributes
1.   For columns which have a few missing entries (less than 50%), we impute them.
2.   To impute them, we should examine the data to see if the columns that have the missing entries categorical or numerical.
- Check the columns if it's from the restricted set (categorical)


In [ ]:
# data examination

print("Column Name              | Data type")
print(X_train.dtypes)

- We have 3 data types
  - float
  - object
  - int
- Use unique() to check if object and int are from restrict set.

In [ ]:
# For int
"""
Cultural_Burn
Flood_Risk_Zone
Invasive_Species_Presence
Litter_Presence
"""
X_train["Litter_Presence"].unique()

1. Cultural_Burn -> category (binary)
2. Flood_Risk_Zone -> category (binary)
3. Invasive_Species_Presence -> category (binary)
4. Litter_Presence -> category (binary)

Every int datatype should be converted to category, and also they are all binary so you don't have to do one-hot encoding for these 4 columns

In [ ]:
# For object
"""
Soil_Type                     object
Burn_Season                   object
Surface_Water_Presence        object
Landform                      object
Human_Disturbance             object
"""
X_train["Human_Disturbance"].unique()

1. Soil_Type -> category (not binary) ['Silty', 'Loam', 'Peaty', 'Clay', 'Sandy', 'Gravel']
2. Burn_Season -> category (not binary) ['Djeran', 'Bunuru', 'Makuru', 'Birak', 'Djilba', 'Kambarang']
3. Surface_Water_Presence -> category (not binary) ['Dry', 'Seasonal', 'Permanent']
4. Landform -> category (not binary)['Hill', 'Flat', 'Valley', 'Ridge', 'Dune']
5. Human_Disturbance -> category (not binary)['Low', 'Medium', 'High']

In [ ]:
#Convert them to category in all datasets(X_train, X_test, X_unseen)
for col in ['Cultural_Burn', 'Flood_Risk_Zone', 'Invasive_Species_Presence', 'Litter_Presence','Soil_Type','Burn_Season', 'Surface_Water_Presence', 'Landform', 'Human_Disturbance']:
  X_train[col] = X_train[col].astype('category')
  X_test[col] = X_test[col].astype('category')
  X_unseen[col] = X_unseen[col].astype('category')

### Impute missing entries

In [ ]:
# compute the fraction of missing values
X_train.isnull().sum()/len(X_train)

We impute the missing values in the column 'Greenness_Index' and 'Days_Since_Burn'.
- both of them are numerical so we impute them with the mean value of the columns

In [ ]:
Greenness_Index_Mean = X_train['Greenness_Index'].mean()
Days_Since_Burn_Mean = X_train['Days_Since_Burn'].mean()
X_train.fillna({'Greenness_Index':Greenness_Index_Mean, 'Days_Since_Burn':Days_Since_Burn_Mean}, inplace=True)

We impute the missing values in X_test and X_unseen with the mean of the column in "X_train". So you should check if there is null value in X_test and X_unseen

- in X_test, there are missing values in the columns
- in X_unseen, it's super clean. there is no missing values
- So, you impute the missing values in X_test **with the mean of the columns in X_train**

In [ ]:
# and do the same on the X_test
X_test.fillna({'Greenness_Index':Greenness_Index_Mean, 'Days_Since_Burn':Days_Since_Burn_Mean}, inplace=True)

## Detect and handle duplicates(both instances and attributes)
Duplicate rows bias your model, so we should check if there are duplicate rows

Duplicate attributes ~~~

Look at df.duplicated() as a starting point, and then combine it with .sum() to see how many rows are duplicated. Unfortunately this only detects adjacent rows that are duplicated.

In [ ]:
# check the duplicate rows in X_train, X_test and X_unseen
X_train.duplicated().sum()

in X_train, there are 37 duplicate rows

in X_test, there are 2 duplicate rows

in X_unseen, it's super clean. There is no duplicate row

In X_train, the fraction of duplicate rows is 0.00925, so it won't cause a big impact, So I won't do anything. I don't touch X_train, therefore I don't do X_test.

- Duplicated attributes

You transpose the dataframe to check if there is duplicated attributes

In [ ]:
# Transpose X_train to check the duplicate columns
X_train_transposed = X_train.T

X_train_transposed.duplicated().sum()

In [ ]:
X_train_transposed

In [ ]:
# Transpose X_test to check the duplicate columns
X_test_transposed = X_test.T

X_test_transposed.duplicated().sum()

In [ ]:
# Transpose X_unseen to check the duplicate columns
X_unseen_transposed = X_unseen.T

X_unseen_transposed.duplicated().sum()

There is no duplicate attributes in X_train, X_teest and X_unseen, so it's all good to move on the next stage!

## Data transformation (scaling/standardization)

In [ ]:
# Detect data distributions for scaling
# Provides summary of numerical value
X_train.describe()

In [ ]:
# Plot the distribution of each numeric attribute
numeric_cols = X_train.select_dtypes(include='number').columns

X_train[numeric_cols].hist(figsize=(15, 15), bins=20)
plt.show()

- Topsoil_Depth, Water_Table_Depth
- Distance_to_Road_M -> Uniform distribution -> min/max
- The rest of them : Normal distribution -> z-score

Topsoil_Depth, Water_Table_Depth should be checked if they don't have much variance.

In [ ]:
plt.scatter(X_train.index, X_train['Topsoil_Depth'])
plt.xlabel('X_train.index')
plt.ylabel('Topsoil_Depth')

In [ ]:
plt.scatter(X_train.index, X_train['Water_Table_Depth'])
plt.xlabel('X_train.index')
plt.ylabel('Water_Table_Depth')

They have a few outliers, but inliers tell me they are not that different. so I'm gonna get rid of these 2 attributes in X_train, X_test and X_unseen

In [ ]:
#Remove Topsoil_Depth, Water_Table_Depth
X_train.drop("Topsoil_Depth",  axis=1, inplace=True)
X_test.drop("Topsoil_Depth", axis=1, inplace=True)
X_unseen.drop("Topsoil_Depth", axis=1, inplace=True)

X_train.drop("Water_Table_Depth",  axis=1, inplace=True)
X_test.drop("Water_Table_Depth", axis=1, inplace=True)
X_unseen.drop("Water_Table_Depth", axis=1, inplace=True)

In [ ]:
# No scaling - binary / categorical columns

# Minmax columns - uniform distributions
minmax = MinMaxScaler()
minmax_cols = ['Distance_to_Road_m']

# Standard scaler columns - other distribution
standard = StandardScaler()
standard_cols = ['NDVI','Leaf_Area_Index','Canopy_Height','Vegetation_Cover_Percent','Greenness_Index','Soil_Moisture','Soil_pH','Soil_Nitrogen_Level','Days_Since_Burn','Fire_Intensity_Score','Regrowth_Indicator','Distance_to_Water_m','Water_Availability_Index','Invasive_Species_Count','Elevation_m','Slope_Degree','Longitude']

# Apply scaling
minmax.fit(X_train[minmax_cols])
standard.fit(X_train[standard_cols])

# Apply min-max to X_train, X_test and X_unseen
X_train[minmax_cols] = minmax.transform(X_train[minmax_cols])
X_test[minmax_cols] = minmax.transform(X_test[minmax_cols])
X_unseen[minmax_cols] = minmax.transform(X_unseen[minmax_cols])

# Apply z-score to X_train, X_test and X_unseen
X_train[standard_cols] = standard.transform(X_train[standard_cols])
X_test[standard_cols] = standard.transform(X_test[standard_cols])
X_unseen[standard_cols] = standard.transform(X_unseen[standard_cols])


In [ ]:
# To verify that it worked
X_train.describe()

In [ ]:
# To double check the distribution after the scaling

# Since the two attributes have been deleted, set the numeric_cols again
numeric_cols = X_train.select_dtypes(include='number').columns
X_train[numeric_cols].hist(figsize=(15, 15), bins=20)
plt.show()

## One-Hot encoding
You do one-hot encoding for ML algorithm to understand

In this dataset, we should do it on the categorical columns below:

- Soil_Type
- Burn_Season
- Surface_Water_Presence
- Landform
- Human_Disturbance

Cultural_Burn, Flood_Risk_Zone, Invasive_Species_Presence and Litter_Presence are category but they are all binary so we don't have to do that on these columns.



In [ ]:
# To convert categorical values to numerical values

# Choose the categorical (also not binary) values
categorical_cols = ['Soil_Type','Burn_Season','Surface_Water_Presence','Landform','Human_Disturbance']

# Build an encoder from the training data
encoder = OneHotEncoder(drop='first',             # Remove one of the categories from each feature (it's redundant)
                        handle_unknown='ignore',  # If there are categories in the test data that are not in the train data, don't complain, just set all values to 0
                        )
encoder.fit(X_train[categorical_cols])

Apply the encoder to X_train, X_test and X_unseen, turning it into a pandas data frame

In [ ]:
# Apply the encoder to X_train and turning it into a pandas data frame
X_train_1hot = pd.DataFrame(encoder.transform(X_train[categorical_cols]).toarray(), columns=encoder.get_feature_names_out())
X_train_1hot.set_index(X_train.index, inplace=True)

# Drop the categorical columns and replace them with the one hot encoded ones
X_train.drop(columns=categorical_cols, inplace=True)
X_train = pd.concat([X_train, X_train_1hot], axis=1)


In [ ]:
# Do the above again for the X_test (apply encoder, replace columns)
X_test_1hot = pd.DataFrame(encoder.transform(X_test[categorical_cols]).toarray(), columns=encoder.get_feature_names_out())
X_test_1hot.set_index(X_test.index, inplace=True)

# Drop the categorical columns and replace them with the one hot encoded ones
X_test.drop(columns=categorical_cols, inplace=True)
X_test = pd.concat([X_test, X_test_1hot], axis=1)

In [ ]:
# Do the above again for the X_unseen (apply encoder, replace columns)
X_unseen_1hot = pd.DataFrame(encoder.transform(X_unseen[categorical_cols]).toarray(), columns=encoder.get_feature_names_out())
X_unseen_1hot.set_index(X_unseen.index, inplace=True)

# Drop the categorical columns and replace them with the one hot encoded ones
X_unseen.drop(columns=categorical_cols, inplace=True)
X_unseen = pd.concat([X_unseen, X_unseen_1hot], axis=1)

# Classification

## Using PCA for dimensionality reduction
This data set has a lot of dimensions, and it's likely that not all the dimensions are needed for a good performance of our model.
You will use PCA as a way to determine the "effective" dimensionality of the data set.
To do this we assume that our data is some combination of signal + noise, and that each contribut some variance to our data.
Requiring that we capture or "explain" 90% of the variance is one way to do this.

In [ ]:
len(X_unseen.columns)

40 dimensions before PCA

In [ ]:
# Do a PCA transform
pca = PCA()
pca.fit(X_train)

# Determine how many components are required to explain 90% of the variance
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(np.cumsum(pca.explained_variance_ratio_))
ax.set_xlabel('number of components')
ax.set_ylabel('cumulative explained variance')
ax.set_xticks(np.arange(0, 40), np.arange(1, 41))
ax.set_ylim([0,1.05])
ax.hlines(y=0.9, xmin=0, xmax=40, linestyles='--')
plt.show()

From the plot above we should identify how many components are required, and then we make a new transform for that number of components.

In [ ]:
# Use PCA to decrease the number of dimensions to the N that we determined
pca = PCA(n_components=0.90)
pca.fit(X_train)

# Transform the data with the selected number of components
X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)
X_unseen_pca = pca.transform(X_unseen)

In [ ]:
# Convert the PCA transformed data to a pandas DataFrame
X_train_pca = pd.DataFrame(X_train_pca, index=X_train.index, columns=[f'C{i+1}' for i in range(X_train_pca.shape[1])])
X_test_pca = pd.DataFrame(X_test_pca, index=X_test.index, columns=[f'C{i+1}' for i in range(X_test_pca.shape[1])])
X_unseen_pca = pd.DataFrame(X_unseen_pca, index=X_unseen.index, columns=[f'C{i+1}' for i in range(X_unseen_pca.shape[1])])

In [ ]:
len(X_unseen_pca.columns)

16 dimensions after PCA

## Class imbalance
We should fix the class imbalance for the performance of out ML model. Before we get to the model building we should check on the class balance.

In [ ]:
# To check the class distribution
import seaborn as sns
sns.histplot(data=vegetation, x='class',hue='class', stat='probability')
plt.show()

To address class imbalance, I'll use SMOTE.

In [ ]:
# SMOTE oversampling
smote = SMOTE(random_state=115)
X_train_smote, y_train_smote = smote.fit_resample(X_train_pca, y_train)

print("SMOTE oversampled dataset shape:", X_train_smote.shape)
print("SMOTE oversampled class distribution:", y_train_smote.value_counts())

##Model training and tuning
Now it's time to try multiple models which are KNN, DT, Naive Bayes and SVM.

### Tuning model hyperparameters with Cross-validation
Our models have hyper-parameters.
I'll select values that produce best average performance with cross validation.

And then I'll select best 2 models and parameters for final 2 models.

#### KNN
Look up the parameters "weights" and "n_neighbours", and choose a range of values over which the GridSearCV will test the model.

In [ ]:
# Choose our classifier
knn = KNeighborsClassifier()
# Choose a cross validation method
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# Set up a range of parameters for our classifier
parameters = {'weights': ['uniform','distance'], # this should be the different weighting schemes
              'n_neighbors':[1,3,7,11,17,21]} # this should be a list of the nearest neigbhours

# Initiate and run the gridsearch
gscv = GridSearchCV(estimator=knn,
                    param_grid=parameters,
                    cv=cv,  # the cross validation folding pattern
                    scoring='accuracy')
gscv.fit(X_train_smote, y_train_smote)

# Use the best parameters to make a new classifier
best_knn = KNeighborsClassifier(n_neighbors=gscv.best_params_['n_neighbors'], weights=gscv.best_params_['weights'])
best_knn.fit(X_train_pca, y_train)

As part of the cross validation we can investigate the mean, standard deviation across the different subsets, as well as the individual values.

In [ ]:
scores = cross_val_score(best_knn, X_train_pca, y_train, cv=cv, scoring='accuracy')
print(f'All scores {scores}')
print(f'Cross-validation Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

Based on the above we would expect that our model would perform well on the test data (80% accuracy).

"""
Cross validation shows us that we should expect an accuracy of about 91% when we run on unseen data.

Let's have a look at the cross validation in detail, and see how the different models worked
"""


In [ ]:
y_hat_knn = best_knn.predict(X_test_pca)
knn_score = accuracy_score(y_test, y_hat_knn)
print(f'Accuracy on test data: {knn_score:.3f}')

##### Performance metric.

We can then look at the confusion matrix to try and explain what is going on with the predictions on the test data

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_hat_knn, display_labels=best_knn.classes_)
plt.show()

In [ ]:
print(classification_report(y_test, y_hat_knn))

여기에 KNN 성능과 관련해서 코멘트

#### Decision Trees

Let's try some other models, starting with a
Decision Tree.

Choose a few values for 'criterion', 'max_depth', and 'min_samples_split', to search over.

In [ ]:
# Choose our classifier
dt = DecisionTreeClassifier()
# Choose a cross validation method
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# Set up a range of parameters for our classifier
parameters = {'criterion': ('gini','entropy'),
              'max_depth': [None, 5, 10],
              'min_samples_split': [2, 5, 10],
              }

# Initiate and run the gridsearch
gscv = GridSearchCV(estimator=dt,
                    param_grid=parameters,
                    cv=cv,  # the cross validation folding pattern
                    scoring='accuracy')
gscv.fit(X_train_smote, y_train_smote)

# Use the best parameters to make a new classifier
best_dt = DecisionTreeClassifier(criterion=gscv.best_params_['criterion'],
                         max_depth=gscv.best_params_['max_depth'],
                         min_samples_split=gscv.best_params_['min_samples_split'])
best_dt.fit(X_train_pca, y_train)

In [ ]:
scores = cross_val_score(best_dt, X_train_pca, y_train, cv=cv, scoring='accuracy')
print(f'All scores {scores}')
print(f'Cross-validation Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
y_hat_dt = best_dt.predict(X_test_pca)
dt_score = accuracy_score(y_test, y_hat_dt)
print(f'Accuracy on test data: {dt_score:.3f}')

##### Performance metric

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_hat_dt, display_labels=best_dt.classes_)
plt.show()

In [ ]:
print(classification_report(y_test, y_hat_dt))

Review the confusion matrix, the cross validaiton peformance, and the test performance for the Decision Tree.
Do you think that this model is suffering from over or under fitting of the data, or does it appear to generalise well?

#### Naive Bayse

In [ ]:
# Choose our classifier
nb = GaussianNB()
# Choose a cross validation method
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Set up a range of parameters for our classifier
parameters = {'priors':[None,]}

# Initiate and run the gridsearch
gscv = GridSearchCV(estimator=nb,
                    param_grid=parameters,
                    cv=cv,  # the cross validation folding pattern
                    scoring='accuracy')
gscv.fit(X_train_smote, y_train_smote)

# Use the best parameters to make a new classifier
best_nb = GaussianNB(priors=gscv.best_params_['priors'])
best_nb.fit(X_train_pca, y_train)


In [ ]:
scores = cross_val_score(best_nb, X_train_pca, y_train, cv=cv, scoring='accuracy')
print(f'All scores {scores}')
print(f'Cross-validation Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
y_hat_nb = best_nb.predict(X_test_pca)
nb_score = accuracy_score(y_test, y_hat_nb)
print(f'Accuracy on test data: {nb_score:.3f}')

##### Performance metric

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_hat_nb, display_labels=best_nb.classes_)
plt.show()

In [ ]:
print(classification_report(y_test, y_hat_nb))

#### SVM

In [ ]:
# Choose our classifier
svc = SVC()
# Choose a cross validation method
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Set up a range of parameters for our classifier
param_grid = {'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']}


# Initiate and run the gridsearch
gscv = GridSearchCV(estimator=svc,
                    param_grid=param_grid,
                    cv=cv,
                    scoring='accuracy')
gscv.fit(X_train_smote, y_train_smote)

# Use the best parameters to make a new classifier
best_svc = SVC(C=gscv.best_params_['C'],
               kernel=gscv.best_params_['kernel'],
               gamma=gscv.best_params_['gamma'])
best_svc.fit(X_train_pca, y_train)


In [ ]:
scores = cross_val_score(best_svc, X_train_pca, y_train, cv=cv, scoring='accuracy')
print(f'All scores {scores}')
print(f'Cross-validation Accuracy: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
y_hat_svc = best_svc.predict(X_test_pca)
svc_score = accuracy_score(y_test, y_hat_svc)
print(f'Accuracy on test data: {svc_score:.3f}')

##### Performance metric

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_hat_svc, display_labels=best_svc.classes_)
plt.show()

In [ ]:
print(classification_report(y_test, y_hat_svc))

## Prediction
Let's try our best 2 models on the unseen dataset!

Best 2 models : Naive Bayes and SVM

In [ ]:
#best_dt.predict(X_test_pca)
final_y_hat_nb = best_nb.predict(X_unseen_pca)
final_y_hat_svc = best_svc.predict(X_unseen_pca)

### Exporting predictions
Once we have made our models and decided which of them we like the best we should save our results to a .csv file to review.


In [ ]:
unseen_dataset

In [ ]:
# Convert the prediction result to Dataframe
best_models_predictions_df = pd.DataFrame({
    "index": unseen_dataset['index'],   # 만약 test 데이터에 ID 컬럼이 있으면 사용
    "Predict1": final_y_hat_nb,
    "Predict2" : final_y_hat_svc
})

# CSV 파일로 저장
best_models_predictions_df.to_csv("Answers_23549670.csv", index=False)
best_models_predictions_df